In [7]:
import time
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

def scrape_dubizzle_realestate(max_pages=50):
    """
    Scrapes Dubizzle Real Estate listings across multiple pages.
    Modified to extract EXACTLY: title, price, location, area, bedrooms, bathrooms.
    """
    
    # 1. Setup Chrome
    chrome_options = Options()
    # chrome_options.add_argument("--headless") # Remove '#' to run invisibly
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)")

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    
    all_properties_data = []
    
    # 2. Loop through the pages
    for page in range(1, max_pages + 1):
        print(f"\n--- Loading Page {page} of {max_pages} ---")
        
        # Dubizzle uses ?page=X in their URLs
        url = f'https://www.dubizzle.com.eg/en/realestate/?page={page}'
        driver.get(url)
        time.sleep(5) 
        
        # 3. The Lazy-Loading 
        print("Scrolling down slowly to load hidden properties...")
        for _ in range(10):
            driver.execute_script("window.scrollBy(0, 800);")
            time.sleep(1.5)
            
        time.sleep(3)
        
        # 4. Grab the listings
        listings = driver.find_elements(By.XPATH, '//article | //li[.//article]')
        print(f"Found {len(listings)} properties on Page {page}.")

        # 5. Extract the data cleanly
        for item in listings:
            try:
                text_lines = item.text.split('\n')
                if len(text_lines) < 3:
                    continue
                    
                raw_text = " | ".join(text_lines)
                
                # Extract Price
                price = next((line for line in text_lines if "EGP" in line), "N/A")
                
                # Extract Title
                title = sorted(text_lines, key=len, reverse=True)[0]
                
                # Extract Beds
                beds_match = re.search(r'(\d+)\s*beds?', raw_text, re.IGNORECASE)
                bedrooms = beds_match.group(1) if beds_match else "N/A"
                
                # Extract Baths
                baths_match = re.search(r'(\d+)\s*baths?', raw_text, re.IGNORECASE)
                bathrooms = baths_match.group(1) if baths_match else "N/A"
                
                # Extract Area
                area_match = re.search(r'(\d+)\s*m²', raw_text, re.IGNORECASE)
                area = area_match.group(1) if area_match else "N/A"
                
                # Extract Location
                location = "N/A"
                if len(text_lines) >= 2:
                    location = text_lines[-2]
                
                # Save ONLY the exact fields requested
                all_properties_data.append({
                    "title": title,
                    "price": price,
                    "location": location,
                    "area": area,
                    "bedrooms": bedrooms,
                    "bathrooms": bathrooms,
                })
            except Exception:
                continue

    # Close the browser when all pages are done
    driver.quit()

    # 6. Save everything to Excel (Notice how it is inside the function)
    if all_properties_data:
        print(f"\nSuccessfully scraped a total of {len(all_properties_data)} properties!")
        df = pd.DataFrame(all_properties_data)
        
        # Drop duplicates based on the new fields
        df = df.drop_duplicates(subset=['title', 'price', 'location']) 
        
        file_name = "Dubizzle_Specific_Fields_Scrape.xlsx"
        df.to_excel(file_name, index=False)
        print(f"All done! Your clean data is saved in: {file_name}")
    else:
        print("\nDidn't find any data. The website structure might have updated.")

# This is the only part outside the function
if __name__ == "__main__":
    scrape_dubizzle_realestate(max_pages=10)


--- Loading Page 1 of 10 ---
Scrolling down slowly to load hidden properties...
Found 4 properties on Page 1.

--- Loading Page 2 of 10 ---
Scrolling down slowly to load hidden properties...
Found 4 properties on Page 2.

--- Loading Page 3 of 10 ---
Scrolling down slowly to load hidden properties...
Found 4 properties on Page 3.

--- Loading Page 4 of 10 ---
Scrolling down slowly to load hidden properties...
Found 4 properties on Page 4.

--- Loading Page 5 of 10 ---
Scrolling down slowly to load hidden properties...
Found 4 properties on Page 5.

--- Loading Page 6 of 10 ---
Scrolling down slowly to load hidden properties...
Found 4 properties on Page 6.

--- Loading Page 7 of 10 ---
Scrolling down slowly to load hidden properties...
Found 4 properties on Page 7.

--- Loading Page 8 of 10 ---
Scrolling down slowly to load hidden properties...
Found 4 properties on Page 8.

--- Loading Page 9 of 10 ---
Scrolling down slowly to load hidden properties...
Found 4 properties on Page 9.

